# SalesInsight PY — Análise e Visualização de Dados de Vendas

## RF01 – Criar ou Carregar o Dataset de Vendas

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]

    data_inicio = datetime(2025, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza ---
        if random.random() < 0.05:
            quantidade = None                    # valor nulo
        if random.random() < 0.04:
            preco = None                         # valor nulo
        if random.random() < 0.06:
            produto = "  " + produto + " "       # espacos extras
        if random.random() < 0.03:
            data_txt = "DATA INVALIDA"           # data invalida
        if random.random() < 0.10:
            cliente = random.choice([            # ruido no nome
                cliente.upper().replace("_", "-"),
                cliente + "!!",
                "  " + cliente,
                cliente.replace("Cliente_", "cliente#"),
            ])

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)


# Gerar e salvar o CSV bruto
df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
print(df_bruto.head())


Dataset gerado com 200 registros.
   id_venda  data_venda      cliente   produto     categoria        regiao  \
0         1  2025-05-21  cliente#016     Mouse   Perifericos       Sudeste   
1         2  2025-09-16  Cliente_039  Notebook  Computadores         Norte   
2         3  2025-03-23  Cliente_045    Tablet     Celulares  Centro-Oeste   
3         4  2025-11-06  Cliente_017  Notebook  Computadores         Norte   
4         5  2025-07-05  Cliente_037    Tablet     Celulares           Sul   

   quantidade  preco_unitario  
0         2.0          102.90  
1         NaN         3204.57  
2         1.0         1939.76  
3         6.0         3864.87  
4        10.0         2008.14  


## RF02 – Inspecionar e Descrever os Dados

In [2]:
def inspecionar_dados(df):
    print("\n=== INSPECAO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    return df

inspecionar_dados(df_bruto)


=== INSPECAO INICIAL DO DATASET ===
Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
produto            0
categoria          0
regiao             0
quantidade        10
preco_unitario     4
dtype: int64

Primeiros registros:
   id_venda  data_venda      cliente   produto     categoria        regiao  \
0         1  2025-05-21  cliente#016     Mouse   Perifericos       Sudeste   
1         2  2025-09-16  Cliente_039  Notebook  Computadores         Norte   
2         3  2025-03-23  Cliente_045    Tablet     Celulares  Centro-Oeste   
3         4  2025-11-06  Cliente_017  Notebook  Comp

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,cliente#016,Mouse,Perifericos,Sudeste,2.0,102.90
1,2,2025-09-16,Cliente_039,Notebook,Computadores,Norte,NaN,3204.57
2,3,2025-03-23,Cliente_045,Tablet,Celulares,Centro-Oeste,1.0,1939.76
3,4,2025-11-06,Cliente_017,Notebook,Computadores,Norte,6.0,3864.87
4,5,2025-07-05,Cliente_037,Tablet,Celulares,Sul,10.0,2008.14
...,...,...,...,...,...,...,...,...
195,196,2025-06-04,CLIENTE-038,Monitor,Computadores,Centro-Oeste,5.0,NaN
196,197,2025-09-15,Cliente_046,Tablet,Celulares,Sul,3.0,1748.92
197,198,2025-08-21,Cliente_048,Smartphone,Celulares,Sul,3.0,2185.60
198,199,2025-11-02,Cliente_006,Headset,Perifericos,Norte,4.0,368.94


## RF03 – Limpar e Tratar os Dados (datetime e regex)

In [3]:
import re

def limpar_dados(df):
    """
    Limpa e trata o DataFrame de vendas.
    Retorna: (df_limpo, relatorio).
    """

    df = df.copy()
    registros_iniciais = len(df)

    # 1. Remover espaços extras das colunas de texto
    colunas_texto = df.select_dtypes(include="object").columns
    for coluna in colunas_texto:
        df[coluna] = df[coluna].str.strip()

    # 2. Converter datas e remover datas inválidas
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    datas_invalidas = df["data_venda"].isna().sum()
    df = df.dropna(subset=["data_venda"])

    # 3. Remover nulos nas colunas críticas
    nulos_criticos = df[["quantidade", "preco_unitario"]].isna().any(axis=1).sum()
    df = df.dropna(subset=["quantidade", "preco_unitario"])

    # 4. Ajustar os tipos numéricos
    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    # 5. Padronizar e sinalizar nomes de clientes
    padrao_cliente = re.compile(r"^Cliente_\d{3}$", flags=re.IGNORECASE)

    df["cliente_fora_padrao"] = ~df["cliente"].apply(
        lambda s: bool(padrao_cliente.fullmatch(str(s)))
    )

    def padronizar_cliente(nome):
        nome_limpo = re.sub(r"[^A-Za-z0-9]", "", str(nome).strip())
        numero = re.search(r"\d{3}", nome_limpo)

        if numero:
            return f"Cliente_{numero.group()}"
        return nome_limpo

    df["cliente"] = df["cliente"].apply(padronizar_cliente)

    # 6. Relatório de limpeza
    registros_finais = len(df)
    registros_removidos = registros_iniciais - registros_finais

    relatorio = {
        "registros_iniciais": registros_iniciais,
        "datas_invalidas_removidas": int(datas_invalidas),
        "nulos_criticos_removidos": int(nulos_criticos),
        "registros_removidos_total": registros_removidos,
        "registros_finais": registros_finais,
        "clientes_fora_padrao": int(df["cliente_fora_padrao"].sum())
    }

    print("\n=== RELATÓRIO DE LIMPEZA ===")
    for chave, valor in relatorio.items():
        print(f"{chave}: {valor}")

    return df, relatorio


df_limpo, relatorio = limpar_dados(df_bruto)


=== RELATÓRIO DE LIMPEZA ===
registros_iniciais: 200
datas_invalidas_removidas: 4
nulos_criticos_removidos: 13
registros_removidos_total: 17
registros_finais: 183
clientes_fora_padrao: 15


In [4]:
df_limpo[["cliente", "cliente_fora_padrao"]].head(20)

,cliente,cliente_fora_padrao
0,Cliente_016,True
2,Cliente_045,False
3,Cliente_017,False
4,Cliente_037,False
5,Cliente_041,False
6,Cliente_011,False
7,Cliente_021,False
8,Cliente_030,False
9,Cliente_033,False
11,Cliente_008,False


## RF04 – Criar Colunas Derivadas com Transformações Condicionais

## RF05 – Calcular Métricas Agregadas com groupby

## RF06 – Segmentar Clientes por Nível de Gasto

## RF07 – Operações Numéricas com NumPy

## RF08 – Criar Visualizações com Matplotlib e Seaborn

## RF09 – Organizar o Código em Funções Reutilizáveis e em uma Classe

## RF10 – Exportar Resultados em CSV e JSON

## RF11 – Executar o Fluxo Completo (Ponto de Entrada)